In [ ]:
%load_ext autoreload
%autoreload 2

# Patent

The `Patent` tool ("DO Patent") extracts chemical structures from a patent or chemistry PDF. It is backed by the platform tool `deeporigin.draco`: it reads the pages of a PDF, detects drawn molecules, and predicts a SMILES string for each one, along with a confidence score and the page it was found on.

`Patent` is an **async** tool. The lifecycle is:

1. Create a `Patent` job from a local `.pdf`.
2. Estimate the cost with `start(quote=True)` and inspect `estimate`.
3. `confirm()` the quoted execution to run it.
4. Track progress with `watch()` (or block with `wait()`).
5. Read the extracted structures with `get_results()`, which returns a `pandas.DataFrame`.

Uploading the PDF to Deep Origin storage happens automatically on `start()`.

In [ ]:
from deeporigin.drug_discovery import Patent
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient()
client

## Create a Patent job

Point `Patent` at a local `.pdf` file. Here we use a one-page example PDF shipped with the client's test fixtures — replace `pdf_path` with the path to your own patent PDF. The file is validated immediately (it must exist and end in `.pdf`), but it is not uploaded until `start()`.

In [ ]:
from pathlib import Path


def find_example_pdf() -> Path:
    """Locate the bundled one-page example patent PDF."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "tests" / "fixtures" / "patent" / "one-page.pdf"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find the example one-page.pdf fixture.")


pdf_path = find_example_pdf()
patent = Patent(pdf=pdf_path)
patent

## Estimate the cost

Call `start(quote=True)` to upload the PDF and request a price without running the job. The instance is left in the `Quoted` state, and the estimated cost is available on `estimate`.

In [ ]:
patent.start(quote=True)
print("status:", patent.status)
print("estimate:", patent.estimate)

## Confirm and run

Once you are happy with the estimate, `confirm()` the execution to submit it. Then `watch()` renders a live progress card in the notebook. In a normal interactive session `await patent.watch()` returns immediately and keeps updating in the background; for a blocking run (for example when executing the notebook headlessly), set `JOB_WATCH_BLOCK=1` or call `await patent.watch(blocking=True)`.

If you'd rather block the cell without the widget, use `patent.wait()` instead.

In [ ]:
patent.confirm()
patent.status

In [ ]:
await patent.watch()

## Retrieve the extracted structures

After the job completes, `get_results()` returns a `pandas.DataFrame` with one row per extracted molecule. Key columns:

- `smiles` — the predicted structure
- `name` — IUPAC name when available
- `page` — the PDF page the molecule was found on
- `confidence` — model confidence for the prediction
- `type`, `record_id`, `source` — provenance metadata

`get_results()` returns `None` until the job has succeeded.

In [ ]:
df = patent.get_results()
df

## Reload an existing run

Every execution has an ID. You can reconstruct a `Patent` object from that ID in a later session and re-fetch its results without re-running the extraction.

In [ ]:
reloaded = Patent.from_id(patent.id)
reloaded.get_results()